In [ ]:
import os
import h5py
d = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_RAWEVENTS/test'#'/data1/ohjinjin/nas_ohjinjin/GOPRO_raw_events/GOPRO_rawevents_rm2/test'

# .h5 파일들의 리스트를 가져옴
h5_files = [f for f in os.listdir(d) if f.endswith('.h5')]

# 각 .h5 파일에 대해 변환 작업 수행
for file_name in h5_files:
    file_path = os.path.join(d, file_name)
    
    # .h5 파일 열기 (읽기 모드)
    with h5py.File(file_path, 'r') as tmpfile:
        print(tmpfile.keys())
        print(tmpfile['events'].keys())
        print(tmpfile['images'].keys()q)

In [ ]:
import os
import numpy as np
import h5py
from PIL import Image

# sharp image만 모아둔 디렉토리
# img_directory = f'/data1/ohjinjin/nas_ohjinjin/GoPro_sharp/test'
img_directory = f'/data1/ohjinjin/GoPro_synthesis/test/'
# EFNet에서 사용했던 h5 확장자의 GoPro with SCER dataset이 저장된 디렉토리
h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_RAWEVENTS/test' #'/data1/ohjinjin/nas_ohjinjin/GOPRO_raw_events/GOPRO_rawevents/test' #'/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_original/GOPRO/test'
# sharp image 두장씩 추가된 새로운 h5을 저장할 디렉토리
output_h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan_raw/test'
if not os.path.exists(output_h5_directory):
    os.makedirs(output_h5_directory)


# scene 별로 저장된 원본 .h5 file의 내용을 그대로 먼저 복사한 후 prev와 next 이미지도 추가해준뒤 새 .h5로 저장해주는 함수
def process_files3(img_directory, h5_directory):
    scene_names = [f for f in os.listdir(img_directory)]
#     print(scene_names)
#     blur_path_list = []; prev_path_list=[]; next_path_list=[];
    
    for scene in scene_names:
        print(f"Current scene is {scene}")
#         datum_idx = [f for f in os.listdir(os.path.join(img_directory, scene))]
# #         print(datum_idx)
#         # EFNet에서 사용한 데이터를 따라 scene마다 가장 처음과 가장 마지막 이미지는 사용하지 않도록 제외
#         for adj_idx, ori_idx in enumerate(datum_idx[1:-1], start=0):
#             blur_path_list.append(os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png'))
#             prev_path_list.append(os.path.join(img_directory, scene, ori_idx, 'left', f'{ori_idx}.png'))
#             next_path_list.append(os.path.join(img_directory, scene, ori_idx, 'right', f'{ori_idx}.png'))
#             path = os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png')
#             image = cv2.imread(path)

#             # OpenCV는 BGR 형식으로 이미지를 읽기 때문에, RGB 형식으로 변환
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#             # 이미지 표시
#             plt.imshow(image)
#             plt.axis('off')  # 축을 표시하지 않음
#             plt.show()
#         break
        
        
        
        old_h5_file_path = os.path.join(h5_directory, scene+".h5")
        new_h5_file_path = os.path.join(output_h5_directory, scene+".h5")
        with h5py.File(old_h5_file_path, 'r') as old_file:
            with h5py.File(new_h5_file_path, 'w') as new_file:
#                 print(old_file.keys())
                for group in old_file.keys():
#                     print(f'COPY GROUPs\n current group: {group}')
                    old_file.copy(group, new_file)
                if 'synthesized_images' not in new_file:
                    blur_group = new_file.create_group('synthesized_images')
                else:
                    blur_group = new_file['synthesized_images']
                    
                if 'sharp_images_prev' not in new_file:
                    prev_group = new_file.create_group('sharp_images_prev')
                else:
                    prev_group = new_file['sharp_images_prev']
                    
                if 'sharp_images_next' not in new_file:
                    next_group = new_file.create_group('sharp_images_next')
                else:
                    next_group = new_file['sharp_images_next']
                
                datum_idx = [f for f in os.listdir(os.path.join(img_directory, scene))]
                # EFNet에서 사용한 데이터를 따라 scene마다 가장 처음과 가장 마지막 이미지는 사용하지 않도록 제외
                for adj_idx, ori_idx in enumerate(datum_idx[1:-1], start=0):
#                     blur_path_list.append(os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png'))
#                     prev_path_list.append(os.path.join(img_directory, scene, ori_idx, 'left', f'{ori_idx}.png'))
#                     next_path_list.append(os.path.join(img_directory, scene, ori_idx, 'right', f'{ori_idx}.png'))
#                 for idx in range(len(blur_path_list)):
                    blur_path = os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png')
                    prev_path = os.path.join(img_directory, scene, ori_idx, 'left', f'{ori_idx}.png')
                    next_path = os.path.join(img_directory, scene, ori_idx, 'right', f'{ori_idx}.png')
                    
                    # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
                    blur_data = np.transpose(np.array(Image.open(blur_path))[:, :, ::-1], (2,0,1))
                    prev_data = np.transpose(np.array(Image.open(prev_path))[:, :, ::-1], (2,0,1))
                    next_data = np.transpose(np.array(Image.open(next_path))[:, :, ::-1], (2,0,1))

                    blur_group.create_dataset(f'image{str(adj_idx).zfill(9)}', data=blur_data)
                    prev_group.create_dataset(f'image{str(adj_idx).zfill(9)}', data=prev_data)
                    next_group.create_dataset(f'image{str(adj_idx).zfill(9)}', data=next_data)
                
#                 # scene마다 가장 처음과 가장 마지막 이미지는 사용할 수 없으므로 제외, EFNet에서 사용한 데이터도 그러함
#                 for index, filename in enumerate(filenames[1:-1], start=0):
#                     prev_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) - 1).zfill(6)+'.png')
#                     next_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) + 1).zfill(6)+'.png')

#                     # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
#                     prev_data = np.transpose(np.array(Image.open(prev_file_path))[:, :, ::-1], (2,0,1))
#                     next_data = np.transpose(np.array(Image.open(next_file_path))[:, :, ::-1], (2,0,1))

#                     prev_group.create_dataset(f'image{str(index).zfill(9)}', data=prev_data)
#                     next_group.create_dataset(f'image{str(index).zfill(9)}', data=next_data)

#     scene_dict = {}

#     # scene별로 sharp image 파일들 분류
#     for file in files:
# #         print(f'Curr file: {file}')
#         scene = '_'.join(file.split('_')[:3])  # 예: 'GOPR0384_11_00'
#         if scene not in scene_dict:
#             scene_dict[scene] = []
#         scene_dict[scene].append(file)

#     for scene, filenames in scene_dict.items():
#         print(f"Current scene is {scene}")
#         filenames.sort()

#         old_h5_file_path = os.path.join(h5_directory, scene+".h5")
#         new_h5_file_path = os.path.join(output_h5_directory, scene+".h5")
#         with h5py.File(old_h5_file_path, 'r') as old_file:
#             with h5py.File(new_h5_file_path, 'w') as new_file:
#                 for group in old_file.keys():
# #                     print(f'COPY GROUPs\n current group: {group}')
#                     old_file.copy(group, new_file)
# #                 print("Contents:", list(new_file.keys()))
#                 if 'sharp_images_prev' not in new_file:
#                     prev_group = new_file.create_group('sharp_images_prev')
#                 else:
#                     prev_group = new_file['sharp_images_prev']
                    
#                 if 'sharp_images_next' not in new_file:
#                     next_group = new_file.create_group('sharp_images_next')
#                 else:
#                     next_group = new_file['sharp_images_next']
                
#                 # scene마다 가장 처음과 가장 마지막 이미지는 사용할 수 없으므로 제외, EFNet에서 사용한 데이터도 그러함
#                 for index, filename in enumerate(filenames[1:-1], start=0):
#                     prev_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) - 1).zfill(6)+'.png')
#                     next_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) + 1).zfill(6)+'.png')

#                     # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
#                     prev_data = np.transpose(np.array(Image.open(prev_file_path))[:, :, ::-1], (2,0,1))
#                     next_data = np.transpose(np.array(Image.open(next_file_path))[:, :, ::-1], (2,0,1))

#                     prev_group.create_dataset(f'image{str(index).zfill(9)}', data=prev_data)
#                     next_group.create_dataset(f'image{str(index).zfill(9)}', data=next_data)

                    
# 파일 처리 시작
process_files3(img_directory, h5_directory)


In [ ]:
# 잘 저장되었는지 확인하기 위한 셀
import matplotlib.pyplot as plt
# import h5py
# import numpy as np
with h5py.File('/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan_raw/test/GOPR0384_11_00.h5', 'r') as file:
    print("Contents:", list(file.keys()))
#     print("c0:", list(file['images'].keys()))
#     print("c0:", np.array(file['images/image000000000']))
#     print("c1:", list(file['masks'].keys()))
#     print("c1:", np.array(file['masks/mask000000001']).shape)
    print("c0:", list(file['events'].keys()))
    print("c1:", list(file['images'].keys()))
    print("c1:", list(file['synthesized_images'].keys()))
    print("c1:", list(file['sharp_images'].keys()))
    print("c2:", list(file['sharp_images_prev'].keys()))
    print("c2:", list(file['sharp_images_next'].keys()))
    
#     print("c2:", np.array(file['sharp_images/image000000001']).shape)
#     print("c3:", list(file['voxels'].keys()))
#     print("c3:", np.array(file['voxels/voxel000000001']).shape)
#     print("c4:", list(file['flows'].keys()))
#     print("c4:", np.array(file['flows/flow000000001']).shape)
    print("c0:", file['events/event000000000'])
    ori_blur_img = np.array(file['images/image000000000'])
    syn_blur_img = np.array(file['synthesized_images/image000000000'])
    img = np.array(file['sharp_images/image000000000'])
    imgprev = np.array(file['sharp_images_prev/image000000000'])
    imgnext = np.array(file['sharp_images_next/image000000000'])
    print(ori_blur_img.shape, syn_blur_img.shape, img.shape)
    plt.title("ori_blur_img")
    plt.imshow(np.transpose(ori_blur_img,(1,2,0))[:, :, [2, 1, 0]])
    plt.show()
    plt.title("syn_blur_img")
    plt.imshow(np.transpose(syn_blur_img,(1,2,0))[:, :, [2, 1, 0]])
    plt.show()
    plt.title("sharp_img")
    plt.imshow(np.transpose(img,(1,2,0))[:, :, [2, 1, 0]])
    plt.show()
    plt.title("imgprev")
    plt.imshow(np.transpose(imgprev,(1,2,0))[:, :, [2, 1, 0]])
    plt.show()
    plt.title("imgnext")
    plt.imshow(np.transpose(imgnext,(1,2,0))[:, :, [2, 1, 0]])
    plt.show()
    
    

In [ ]:
import os
import numpy as np
import h5py
from PIL import Image

# sharp image만 모아둔 디렉토리
# img_directory = f'/data1/ohjinjin/nas_ohjinjin/GoPro_sharp/test'
img_directory = f'/data1/ohjinjin/GoPro_synthesis/train/'
# EFNet에서 사용했던 h5 확장자의 GoPro with SCER dataset이 저장된 디렉토리
h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_RAWEVENTS/train' #'/data1/ohjinjin/nas_ohjinjin/GOPRO_raw_events/GOPRO_rawevents/train'#'/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_original/GOPRO/train'
# sharp image 두장씩 추가된 새로운 h5을 저장할 디렉토리
output_h5_directory = '/data1/ohjinjin/nas_ohjinjin/GOPRO_EFNet_jinjin/GOPRO_joowan_raw/train'
if not os.path.exists(output_h5_directory):
    os.makedirs(output_h5_directory)


# scene 별로 저장된 원본 .h5 file의 내용을 그대로 먼저 복사한 후 prev와 next 이미지도 추가해준뒤 새 .h5로 저장해주는 함수
def process_files3(img_directory, h5_directory):
    scene_names = [f for f in os.listdir(img_directory)]
#     print(scene_names)
#     blur_path_list = []; prev_path_list=[]; next_path_list=[];
    
    for scene in scene_names:
        print(f"Current scene is {scene}")
#         datum_idx = [f for f in os.listdir(os.path.join(img_directory, scene))]
# #         print(datum_idx)
#         # EFNet에서 사용한 데이터를 따라 scene마다 가장 처음과 가장 마지막 이미지는 사용하지 않도록 제외
#         for adj_idx, ori_idx in enumerate(datum_idx[1:-1], start=0):
#             blur_path_list.append(os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png'))
#             prev_path_list.append(os.path.join(img_directory, scene, ori_idx, 'left', f'{ori_idx}.png'))
#             next_path_list.append(os.path.join(img_directory, scene, ori_idx, 'right', f'{ori_idx}.png'))
#             path = os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png')
#             image = cv2.imread(path)

#             # OpenCV는 BGR 형식으로 이미지를 읽기 때문에, RGB 형식으로 변환
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#             # 이미지 표시
#             plt.imshow(image)
#             plt.axis('off')  # 축을 표시하지 않음
#             plt.show()
#         break
        
        
        
        old_h5_file_path = os.path.join(h5_directory, scene+".h5")
        new_h5_file_path = os.path.join(output_h5_directory, scene+".h5")
        with h5py.File(old_h5_file_path, 'r') as old_file:
            with h5py.File(new_h5_file_path, 'w') as new_file:
#                 print(old_file.keys())
                for group in old_file.keys():
#                     print(f'COPY GROUPs\n current group: {group}')
                    old_file.copy(group, new_file)
                if 'synthesized_images' not in new_file:
                    blur_group = new_file.create_group('synthesized_images')
                else:
                    blur_group = new_file['synthesized_images']
                    
                if 'sharp_images_prev' not in new_file:
                    prev_group = new_file.create_group('sharp_images_prev')
                else:
                    prev_group = new_file['sharp_images_prev']
                    
                if 'sharp_images_next' not in new_file:
                    next_group = new_file.create_group('sharp_images_next')
                else:
                    next_group = new_file['sharp_images_next']
                
                datum_idx = [f for f in os.listdir(os.path.join(img_directory, scene))]
                # EFNet에서 사용한 데이터를 따라 scene마다 가장 처음과 가장 마지막 이미지는 사용하지 않도록 제외
                for adj_idx, ori_idx in enumerate(datum_idx[1:-1], start=0):
#                     blur_path_list.append(os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png'))
#                     prev_path_list.append(os.path.join(img_directory, scene, ori_idx, 'left', f'{ori_idx}.png'))
#                     next_path_list.append(os.path.join(img_directory, scene, ori_idx, 'right', f'{ori_idx}.png'))
#                 for idx in range(len(blur_path_list)):
                    blur_path = os.path.join(img_directory, scene, ori_idx, 'blur', f'{ori_idx}.png')
                    prev_path = os.path.join(img_directory, scene, ori_idx, 'left', f'{ori_idx}.png')
                    next_path = os.path.join(img_directory, scene, ori_idx, 'right', f'{ori_idx}.png')
                    
                    # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
                    blur_data = np.transpose(np.array(Image.open(blur_path))[:, :, ::-1], (2,0,1))
                    prev_data = np.transpose(np.array(Image.open(prev_path))[:, :, ::-1], (2,0,1))
                    next_data = np.transpose(np.array(Image.open(next_path))[:, :, ::-1], (2,0,1))

                    blur_group.create_dataset(f'image{str(adj_idx).zfill(9)}', data=blur_data)
                    prev_group.create_dataset(f'image{str(adj_idx).zfill(9)}', data=prev_data)
                    next_group.create_dataset(f'image{str(adj_idx).zfill(9)}', data=next_data)
                
#                 # scene마다 가장 처음과 가장 마지막 이미지는 사용할 수 없으므로 제외, EFNet에서 사용한 데이터도 그러함
#                 for index, filename in enumerate(filenames[1:-1], start=0):
#                     prev_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) - 1).zfill(6)+'.png')
#                     next_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) + 1).zfill(6)+'.png')

#                     # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
#                     prev_data = np.transpose(np.array(Image.open(prev_file_path))[:, :, ::-1], (2,0,1))
#                     next_data = np.transpose(np.array(Image.open(next_file_path))[:, :, ::-1], (2,0,1))

#                     prev_group.create_dataset(f'image{str(index).zfill(9)}', data=prev_data)
#                     next_group.create_dataset(f'image{str(index).zfill(9)}', data=next_data)

#     scene_dict = {}

#     # scene별로 sharp image 파일들 분류
#     for file in files:
# #         print(f'Curr file: {file}')
#         scene = '_'.join(file.split('_')[:3])  # 예: 'GOPR0384_11_00'
#         if scene not in scene_dict:
#             scene_dict[scene] = []
#         scene_dict[scene].append(file)

#     for scene, filenames in scene_dict.items():
#         print(f"Current scene is {scene}")
#         filenames.sort()

#         old_h5_file_path = os.path.join(h5_directory, scene+".h5")
#         new_h5_file_path = os.path.join(output_h5_directory, scene+".h5")
#         with h5py.File(old_h5_file_path, 'r') as old_file:
#             with h5py.File(new_h5_file_path, 'w') as new_file:
#                 for group in old_file.keys():
# #                     print(f'COPY GROUPs\n current group: {group}')
#                     old_file.copy(group, new_file)
# #                 print("Contents:", list(new_file.keys()))
#                 if 'sharp_images_prev' not in new_file:
#                     prev_group = new_file.create_group('sharp_images_prev')
#                 else:
#                     prev_group = new_file['sharp_images_prev']
                    
#                 if 'sharp_images_next' not in new_file:
#                     next_group = new_file.create_group('sharp_images_next')
#                 else:
#                     next_group = new_file['sharp_images_next']
                
#                 # scene마다 가장 처음과 가장 마지막 이미지는 사용할 수 없으므로 제외, EFNet에서 사용한 데이터도 그러함
#                 for index, filename in enumerate(filenames[1:-1], start=0):
#                     prev_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) - 1).zfill(6)+'.png')
#                     next_file_path = os.path.join(img_directory, scene + '_' + str(int(filename[-10:-4]) + 1).zfill(6)+'.png')

#                     # 추가할 sharp image들 열어서 기존 h5파일에 이미지들 저장해둔 차원과 통일시켜주기
#                     prev_data = np.transpose(np.array(Image.open(prev_file_path))[:, :, ::-1], (2,0,1))
#                     next_data = np.transpose(np.array(Image.open(next_file_path))[:, :, ::-1], (2,0,1))

#                     prev_group.create_dataset(f'image{str(index).zfill(9)}', data=prev_data)
#                     next_group.create_dataset(f'image{str(index).zfill(9)}', data=next_data)

                    
# 파일 처리 시작
process_files3(img_directory, h5_directory)
